# Profiling an Ambiq Application with `hpx`

This notebook is a beginner-friendly walkthrough of **heliaPROFILER** (`hpx`) through its typed Python API. It uses an immutable `hpx.Session` so configuration can be preserved and branched across experiments without invoking the CLI through subprocesses.

By the end, you will know how to:

- Check that your machine can build, flash, and capture profiling data.
- Profile a model with the `helia-rt` interpreter engine.
- Profile the same model with the `helia-aot` ahead-of-time compiler.
- Inspect typed profile results and generated artifacts.
- Compare two runs and identify the layers that changed most.
- Locate overlays for Model Explorer.

> Hardware profiling cells are guarded by a `RUN_HARDWARE` flag so you can explore the programmatic API before flashing a board.

## 0. Mental Model

An `hpx.Session` is an immutable description of a profiling environment. Configure a base session once, derive independent branches for each engine or experiment, and call `Session.profile()` to run one profiling experiment:

1. Resolve the model, selected engine, target, and profiling options.
2. Generate a temporary NSX firmware project.
3. Build and flash that firmware to an Apollo EVB.
4. Capture PMU counters layer-by-layer.
5. Return a typed `ProfileResult` and write structured result files.

The most common engines are:

| Engine | What it means | When to use it |
| --- | --- | --- |
| `helia-rt` | Runtime interpreter path based on Ambiq's optimized TFLM fork | Baseline profiling, debugging, flexible model iteration |
| `helia-aot` | Ahead-of-time compiler that emits model-specific C code | Production-style performance and smaller runtime footprint |

A good workflow is usually:

1. Create one base session for the model and target.
2. Derive a `helia-rt` session and inspect its typed result.
3. Derive a `helia-aot` session from the same base.
4. Compare the two returned results.

## 1. Configure This Notebook

Edit these values for your setup. The defaults assume this notebook is inside the `helia-profiler` repo and uses the included quickstart keyword-spotting model.

In [ ]:
from pathlib import Path
import os
import helia_profiler as hpx

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the helia-profiler repository root")

MODEL = REPO_ROOT / "examples" / "quickstart" / "kws_model.tflite"
BOARD = "apollo510_evb"
TOOLCHAIN = "arm-none-eabi-gcc"
TRANSPORT = "rtt"

RESULTS_ROOT = REPO_ROOT / "results" / "notebook_walkthrough"
RT_RESULTS = RESULTS_ROOT / "helia_rt"
AOT_RESULTS = RESULTS_ROOT / "helia_aot"
COMPARE_RESULTS = RESULTS_ROOT / "rt_vs_aot"

# Set this to True only when your EVB, J-Link, toolchain, and RTT sources are ready.
RUN_HARDWARE = False

EXTRA_PATHS = [
    "/Applications/ArmGNUToolchain/15.2.rel1/arm-none-eabi/bin",
    "/Applications/SEGGER/JLink_V948",
    "/Applications/SEGGER/JLink",
]
path_entries = [str(path) for path in EXTRA_PATHS if Path(path).exists()]
os.environ["PATH"] = os.pathsep.join(path_entries + [os.environ.get("PATH", "")])

SEGGER_RTT_PATH = REPO_ROOT / ".deps" / "SEGGER_RTT"
if SEGGER_RTT_PATH.exists():
    os.environ["SEGGER_RTT_PATH"] = str(SEGGER_RTT_PATH)

base_session = (
    hpx.Session()
    .with_model(MODEL)
    .with_target(board=BOARD, toolchain=TOOLCHAIN, transport=TRANSPORT)
    .with_profiling(iterations=100, warmup=5)
)

print(f"Repo root: {REPO_ROOT}")
print(f"Model:     {MODEL}")
print(f"Results:   {RESULTS_ROOT}")
print(f"Hardware cells enabled: {RUN_HARDWARE}")

Repo root: /home/user/helia-profiler
Model:     /home/user/helia-profiler/examples/quickstart/kws_model.tflite
Results:   /home/user/helia-profiler/results/notebook_walkthrough
Hardware cells enabled: True


In [ ]:
# Session operations below call the typed Python API directly.

## 2. Check Your Installation

`Session.doctor()` returns structured checks for the external tools and Python packages needed for profiling. Required items include CMake, Ninja, an Arm toolchain, J-Link tools, and the `neuralspotx` Python package.

Optional items, such as `helia-aot` and Arm Compiler, are useful for specific workflows but not required for every `helia-rt` run.

In [ ]:
doctor_result = base_session.show(base_session.doctor())

$ uv run hpx doctor

Toolchain Check                                                                 
╭────┬────────────────────────────────────┬────────────────────────────────────╮
│    │ Tool                               │ Path                               │
├────┼────────────────────────────────────┼────────────────────────────────────┤
│ ✓  │ ARM GCC toolchain                  │ /Applications/ArmGNUToolchain/15.… │
│ ✓  │ CMake (>= 3.24)                    │ /etc/profiles/per-user/user… │
│ ✓  │ Ninja build system                 │ /etc/profiles/per-user/user… │
│ ✓  │ SEGGER J-Link commander            │ /Applications/SEGGER/JLink_V948/J… │
│ ✓  │ neuralspotx Python package         │ installed                          │
│ ✓  │ pylink Python package (RTT/SWO     │ installed                          │
│    │ transport)                         │                                    │
│ ✓  │ heliaAOT compiler                  │ installed                          │
│ –  │ ARM Compiler

CompletedProcess(args='uv run hpx doctor', returncode=0, stdout='\x1b\x1bwarning\x1b\x1b\x1b:\x1b \x1b`VIRTUAL_ENV=/home/user/.cache/uv/builds-v0/.tmpqFGc30` does not match the project environment path `.venv` and will be ignored; use `--active` to target the active environment instead\x1b\n\n\x1bToolchain Check\x1b\x1b                                                                 \x1b\n╭────┬────────────────────────────────────┬────────────────────────────────────╮\n│\x1b \x1b\x1b  \x1b\x1b \x1b│\x1b \x1b\x1bTool                              \x1b\x1b \x1b│\x1b \x1b\x1bPath                              \x1b\x1b \x1b│\n├────┼────────────────────────────────────┼────────────────────────────────────┤\n│ \x1b✓\x1b  │ ARM GCC toolchain                  │\x1b \x1b\x1b/Applications/ArmGNUToolchain/15.…\x1b\x1b \x1b│\n│ \x1b✓\x1b  │ CMake (>= 3.24)                    │\x1b \x1b\x1b/etc/profiles/per-user/user…\x1b\x1b \x1b│\n│ \x1b✓\x1b  │ Ninja build system                 │\x1b \x1b\x1b/etc

`Session.boards()` returns the typed board definitions. Use a board name in `with_target(board=...)`.

In [ ]:
boards = base_session.show(base_session.boards())

$ uv run hpx boards
 Board                    SoC            Core           Backends             Dom
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 apollo3p_evb             apollo3p       cortex-m4      dwt                  cpu
 apollo3p_evb_cygnus      apollo3p       cortex-m4      dwt                  cpu
 apollo4p_evb             apollo4p       cortex-m4      dwt                  cpu
 apollo4l_evb             apollo4l       cortex-m4      dwt                  cpu
 apollo4l_blue_evb        apollo4l       cortex-m4      dwt                  cpu
 apollo4p_blue_kbr_evb    apollo4p       cortex-m4      dwt                  cpu
 apollo4p_blue_kxr_evb    apollo4p       cortex-m4      dwt                  cpu
 apollo510_evb            apollo510      cortex-m55     dwt, armv8m-pmu      cpu
                                                                             mve
 apollo510b_evb           apollo510b     cortex-m55     dwt, armv8m-pmu      cpu
        

CompletedProcess(args='uv run hpx boards', returncode=0, stdout='\x1b\x1bwarning\x1b\x1b\x1b:\x1b \x1b`VIRTUAL_ENV=/home/user/.cache/uv/builds-v0/.tmpqFGc30` does not match the project environment path `.venv` and will be ignored; use `--active` to target the active environment instead\x1b\n\x1b \x1b\x1bBoard                 \x1b\x1b \x1b \x1b \x1b\x1bSoC         \x1b\x1b \x1b \x1b \x1b\x1bCore        \x1b\x1b \x1b \x1b \x1b\x1bBackends          \x1b\x1b \x1b \x1b \x1b\x1bDom\x1b\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n apollo3p_evb             apollo3p       cortex-m4      dwt                  cpu\n apollo3p_evb_cygnus      apollo3p       cortex-m4      dwt                  cpu\n apollo4p_evb             apollo4p       cortex-m4      dwt                  cpu\n apollo4l_evb             apollo4l       cortex-m4      dwt                  cpu\n apollo4l_blue_evb        apollo4l       cortex-m4      dwt                  cpu\n apollo4p_blue_kbr_evb

## 3. Analyze the Model Without Hardware

`Session.analyze()` returns a typed compute breakdown without building or flashing firmware. This confirms that the model can be inspected and gives a rough sense of which layers may be expensive.

In [ ]:
rt_analysis = None
try:
    rt_analysis = base_session.with_engine("helia-rt").analyze()
except hpx.HpxError as exc:
    print(exc)
else:
    print(f"Engine: {rt_analysis.engine}")
    print(f"Layers: {len(rt_analysis.layers)}")
    print(f"Total MACs: {rt_analysis.total_macs:,}")
    print(f"Total operations: {rt_analysis.total_ops:,}")

$ uv run hpx analyze /home/user/helia-profiler/examples/quickstart/kws_model.tflite

────────────────────── Model Analysis — kws_model.tflite ───────────────────────

  Layers        13         
  Total MACs    2,656,768  
  Total OPS     5,321,596  
  Parameters    22,604     

Per-Layer Breakdown                                                             
    #   Operator                           MACs              OPS     % MACs   Ou
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    0   CONV_2D                         320,000          640,000      12.0%   [1
    1   DEPTHWISE_CONV_2D                72,000          144,000       2.7%   [1
    2   CONV_2D                         512,000        1,024,000      19.3%   [1
    3   DEPTHWISE_CONV_2D                72,000          144,000       2.7%   [1
    4   CONV_2D                         512,000        1,024,000      19.3%   [1
    5   DEPTHWISE_CONV_2D                72,000          144,000       2

CompletedProcess(args='uv run hpx analyze /home/user/helia-profiler/examples/quickstart/kws_model.tflite', returncode=0, stdout='\x1b\x1bwarning\x1b\x1b\x1b:\x1b \x1b`VIRTUAL_ENV=/home/user/.cache/uv/builds-v0/.tmpqFGc30` does not match the project environment path `.venv` and will be ignored; use `--active` to target the active environment instead\x1b\n\n\x1b────────────────────── \x1b\x1bModel Analysis — kws_model.tflite\x1b\x1b ───────────────────────\x1b\n\n\x1b  \x1b\x1bLayers    \x1b\x1b  \x1b  13         \n\x1b  \x1b\x1bTotal MACs\x1b\x1b  \x1b  \x1b2,656,768\x1b  \n\x1b  \x1b\x1bTotal OPS \x1b\x1b  \x1b  5,321,596  \n\x1b  \x1b\x1bParameters\x1b\x1b  \x1b  22,604     \n\n\x1bPer-Layer Breakdown\x1b\x1b                                                             \x1b\n\x1b \x1b\x1b   #\x1b\x1b \x1b \x1b \x1b\x1bOperator              \x1b\x1b \x1b \x1b \x1b\x1b          MACs\x1b\x1b \x1b \x1b \x1b\x1b           OPS\x1b\x1b \x1b \x1b \x1b\x1b  % MACs\x1b\x1b \x1b \x1b \x1b\x1bOu\x

Derive an AOT session to analyze the graph after `helia-aot` transformations. Keeping the RT and AOT analyses as separate values makes them easy to inspect or compare programmatically.

In [ ]:
aot_analysis = None
try:
    aot_analysis = base_session.with_engine("helia-aot").analyze()
except hpx.HpxError as exc:
    print(exc)
else:
    print(f"Engine: {aot_analysis.engine}")
    print(f"Layers: {len(aot_analysis.layers)}")
    print(f"Total MACs: {aot_analysis.total_macs:,}")
    print(f"Total operations: {aot_analysis.total_ops:,}")

$ uv run hpx analyze /home/user/helia-profiler/examples/quickstart/kws_model.tflite --engine helia-aot --board apollo510_evb --compare
WARNING   unresolved dynamic tensors remain (14): ['0', '22', '23', '24', '25', 
         '26', '27', '28', '29', '30', '31', '32', '33', '34']. Conversion      
         continues with placeholder dims.                                       

──────────────── Model Analysis — kws_model.tflite  (helia-aot) ────────────────

  Engine        helia-aot  
  Layers        13         
  Total MACs    2,656,768  
  Total OPS     5,321,596  
  Parameters    22,016     

Per-Layer Breakdown                                                             
    #    Src   Operator                           MACs              OPS     % MA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    0      0   CONV_2D                         320,000          640,000      12.
    1      1   DEPTHWISE_CONV_2D                72,000          144,000   

CompletedProcess(args='uv run hpx analyze /home/user/helia-profiler/examples/quickstart/kws_model.tflite --engine helia-aot --board apollo510_evb --compare', returncode=0, stdout="\x1b\x1bwarning\x1b\x1b\x1b:\x1b \x1b`VIRTUAL_ENV=/home/user/.cache/uv/builds-v0/.tmpqFGc30` does not match the project environment path `.venv` and will be ignored; use `--active` to target the active environment instead\x1b\n\x1bWARNING \x1b  unresolved dynamic tensors remain \x1b(\x1b\x1b14\x1b\x1b)\x1b: \x1b[\x1b\x1b'0'\x1b, \x1b'22'\x1b, \x1b'23'\x1b, \x1b'24'\x1b, \x1b'25'\x1b, \n         \x1b'26'\x1b, \x1b'27'\x1b, \x1b'28'\x1b, \x1b'29'\x1b, \x1b'30'\x1b, \x1b'31'\x1b, \x1b'32'\x1b, \x1b'33'\x1b, \x1b'34'\x1b\x1b[1m]\x1b. Conversion      \n         continues with placeholder dims.                                       \n\n\x1b──────────────── \x1b\x1bModel Analysis — kws_model.tflite\x1b  \x1b(helia-aot)\x1b\x1b ────────────────\x1b\n\n\x1b  \x1b\x1bEngine    \x1b\x1b  \x1b  \x1bhelia-aot\x1b  \n\x1b 

## 4. Profile with `helia-rt`

`helia-rt` is the default interpreter-style engine. Derive an RT session from the shared base, add its engine-specific arena size and output directory, then call `profile()` when hardware execution is enabled.

In [ ]:
rt_session = (
    base_session
    .with_engine("helia-rt")
    .with_output(dir=RT_RESULTS)
    .with_model(MODEL, arena_size=131_072)
)

rt_result = None
if RUN_HARDWARE:
    rt_result = rt_session.profile()
    print("Reports:")
    print(*rt_result.report_paths, sep="\n")
else:
    print("Skipped. Set RUN_HARDWARE = True in the setup cell to run this profile.")

uv run hpx profile /home/user/helia-profiler/examples/quickstart/kws_model.tflite --engine helia-rt --board apollo510_evb --toolchain arm-none-eabi-gcc --transport rtt --arena-size 131072 --iterations 100 --warmup 5 --output-dir /home/user/helia-profiler/results/notebook_walkthrough/helia_rt
$ uv run hpx profile /home/user/helia-profiler/examples/quickstart/kws_model.tflite --engine helia-rt --board apollo510_evb --toolchain arm-none-eabi-gcc --transport rtt --arena-size 131072 --iterations 100 --warmup 5 --output-dir /home/user/helia-profiler/results/notebook_walkthrough/helia_rt
⠋ ━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌  27%  ▸  resolve_jlink_probe (3/11)
⠙ ━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌  27%  ▸  resolve_jlink_probe (3/11)
⠹ ━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌  27%  ▸  resolve_jlink_probe (3/11)
⠸ ━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌  27%  ▸  resolve_jlink_probe (3/11)
⠼ ━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌  27%  ▸  resolve_jlink_probe (3/11)
⠇ ━━━━━━━━━━━━━━━━━━╌╌  90%  📊  Capture PMU (10/11)
⠏ ━━━━━━━━━━━━━━━━━━╌╌  90%  📊  Capture PMU (10/11)
⠋ ━━━━━━━━━━━

## 5. Inspect Results and Artifacts

`Session.profile()` returns a typed `ProfileResult` for immediate inspection and also writes durable artifacts:

| File | Purpose |
| --- | --- |
| `summary.json` | Fast overview: total cycles, layer count, top layers, memory, binary size |
| `profile_results.csv` | One row per layer, with PMU counters and cycle counts |
| `run_metadata.json` | Provenance: model hash, board, toolchain, engine, config, firmware details |

Use `result.layers`, `result.metadata`, and convenience properties in the notebook; use `result.report_paths` to locate persisted files.

In [ ]:
def show_profile(result):
    print(f"Layers: {result.layer_count}")
    print(f"Total cycles: {result.total_cycles:,.0f}")
    print(f"Overflow detected: {result.overflow_detected}")

    if result.metadata.platform:
        print(f"Board: {result.metadata.platform.board}")
        print(f"SoC: {result.metadata.platform.soc}")

    print("Reports:")
    print(*result.report_paths, sep="\n")
    return result


rt_summary = show_profile(rt_result) if rt_result is not None else None

{
  "engine": "helia-rt",
  "layers": 13,
  "total_cycles": 2029914.7,
  "overflow_detected": false,
  "memory": {
    "arena_size": 131072,
    "allocated_arena": 29780,
    "model_size": 53744,
    "num_tensors": 35,
    "input_size": 490,
    "output_size": 12
  },
  "binary": {
    "text": 265748,
    "data": 154732,
    "bss": 353168,
    "total": 773648
  }
}


In [ ]:
def show_rows(rows, columns):
    if not rows:
        print("No rows to show")
        return

    widths = {column: len(column) for column in columns}
    for row in rows:
        for column in columns:
            widths[column] = max(widths[column], len(str(row.get(column, ""))))

    print("  ".join(column.ljust(widths[column]) for column in columns))
    print("  ".join("-" * widths[column] for column in columns))
    for row in rows:
        print("  ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))


def top_layers(result, n=10):
    if result is None:
        print("No profile result available")
        return []

    layers = sorted(result.layers, key=lambda layer: layer.cycles or 0, reverse=True)[:n]
    rows = [
        {
            "id": layer.id,
            "op": layer.op,
            "cycles": layer.cycles,
            "overflow": layer.overflow,
            **layer.counters,
        }
        for layer in layers
    ]
    columns = ["id", "op", "cycles", "overflow"]
    counter_names = sorted(
        name for row in rows for name in row if name not in columns
    )
    show_rows(rows, columns + counter_names)
    return layers


rt_layers = top_layers(rt_result)

id  op                 cycles     ARM_PMU_CPU_CYCLES  ARM_PMU_INST_RETIRED  overflow
--  -----------------  ---------  ------------------  --------------------  --------
0   CONV_2D            343616.06  343616.06           262269.0              False   
2   CONV_2D            209921.05  209921.05           164528.0              False   
6   CONV_2D            209916.84  209916.84           164528.0              False   
8   CONV_2D            209916.39  209916.39           164528.0              False   
4   CONV_2D            209913.42  209913.42           164528.0              False   
1   DEPTHWISE_CONV_2D  207129.38  207129.38           155688.0              False   
3   DEPTHWISE_CONV_2D  206964.42  206964.42           155688.0              False   
5   DEPTHWISE_CONV_2D  206953.25  206953.25           155688.0              False   
7   DEPTHWISE_CONV_2D  206946.4   206946.4            155688.0              False   
9   AVERAGE_POOL_2D    14886.51   14886.51            9071.0     

## 6. Profile with `helia-aot`

`helia-aot` compiles the model ahead of time into model-specific C code. It may have different performance, memory, and binary-size characteristics than `helia-rt`.

Install support with:

```bash
uv sync --extra aot
```

or, for an installed package:

```bash
pip install 'helia-profiler[aot]'
```

In [ ]:
aot_session = (
    base_session
    .with_engine("helia-aot")
    .with_output(dir=AOT_RESULTS)
)

aot_result = None
if RUN_HARDWARE:
    aot_result = aot_session.profile()
    print("Reports:")
    print(*aot_result.report_paths, sep="\n")
else:
    print("Skipped. Set RUN_HARDWARE = True in the setup cell to run this profile.")

uv run hpx profile /home/user/helia-profiler/examples/quickstart/kws_model.tflite --engine helia-aot --board apollo510_evb --toolchain arm-none-eabi-gcc --transport rtt --iterations 100 --warmup 5 --output-dir /home/user/helia-profiler/results/notebook_walkthrough/helia_aot
$ uv run hpx profile /home/user/helia-profiler/examples/quickstart/kws_model.tflite --engine helia-aot --board apollo510_evb --toolchain arm-none-eabi-gcc --transport rtt --iterations 100 --warmup 5 --output-dir /home/user/helia-profiler/results/notebook_walkthrough/helia_aot
⠋ ━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌  27%  ▸  resolve_jlink_probe (3/11)
⠙ ━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌  27%  ▸  resolve_jlink_probe (3/11)WARNING   unresolved dynamic tensors remain (14): ['0', '22', '23', '24', '25', 
         '26', '27', '28', '29', '30', '31', '32', '33', '34']. Conversion      
         continues with placeholder dims.                                       

⠹ ━━━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌  36%  ⚙️  Prepare engine (4/11)
⠸ ━━━━━━━╌╌╌╌╌╌╌╌╌╌╌╌╌  36%  ⚙️ 

In [ ]:
aot_summary = show_profile(aot_result) if aot_result is not None else None
aot_layers = top_layers(aot_result)

## 7. Compare Runs

Once both profiles complete, `Session.compare()` accepts the two `ProfileResult` values directly and returns a typed `CompareResult`. Passing `output_dir` also writes `compare_summary.json` and `layer_diff.csv`.

In [ ]:
compare_result = None

if rt_result is not None and aot_result is not None:
    compare_result = base_session.compare(
        rt_result,
        aot_result,
        output_dir=COMPARE_RESULTS,
    )
    print(f"Compared {len(compare_result.layer_rows)} layers")
    print(f"Warnings: {len(compare_result.warnings)}")
else:
    print(
        "Skipped. Run both profiles first, or use existing result directories "
        "with base_session.compare(RT_RESULTS, AOT_RESULTS)."
    )

In [ ]:
if compare_result is not None:
    top_diffs = sorted(
        compare_result.layer_rows,
        key=lambda row: abs(row.delta_cycles or 0),
        reverse=True,
    )[:10]

    for row in top_diffs:
        print(
            f"{row.id}: {row.baseline_op} -> {row.candidate_op}; "
            f"{row.baseline_cycles} -> {row.candidate_cycles}; "
            f"delta={row.delta_cycles}; speedup={row.speedup}"
        )
else:
    print("No comparison result available.")

## 8. Model Explorer Overlays

`hpx` writes Model Explorer overlay files by default under `model_explorer/`. These let you load the original `.tflite` graph and color nodes by profiler counters.

Workflow:

1. Open [Model Explorer](https://google-ai-edge.github.io/model-explorer/).
2. Load your `.tflite` model.
3. Click **Add Overlay**.
4. Select a file such as `me_overlay_ARM_PMU_CPU_CYCLES.json` from the run's `model_explorer/` directory.

Red nodes are the highest values for that counter; green nodes are lower values.

In [ ]:
overlay_paths = []

if rt_result is not None:
    overlay_paths = [
        path for path in rt_result.report_paths if path.name.startswith("me_overlay_")
    ]

if overlay_paths:
    print(*overlay_paths[:10], sep="\n")
else:
    overlay_dir = RT_RESULTS / "model_explorer"
    print(f"No overlays found yet in {overlay_dir}")

## 9. Optional: Power Profiling

Power capture is another immutable branch of the same base session. With a Joulescope JS110/JS220 connected, enable power and profile normally:

```python
power_session = (
    base_session
    .with_engine("helia-rt")
    .with_power(enabled=True, duration_s=10)
    .with_output(dir=RESULTS_ROOT / "power_run")
)
power_result = power_session.profile() if RUN_HARDWARE else None
```

Power metrics are available through `power_result.power` and in the generated reports.

## 10. Troubleshooting Checklist

| Symptom | What to check |
| --- | --- |
| Doctor checks cannot find ARM GCC | Add the `arm-none-eabi` toolchain `bin/` directory to `PATH`. |
| J-Link not found | Add the SEGGER J-Link directory to `PATH`, or install SEGGER J-Link tools. |
| `SEGGER_RTT_PATH is not set` | Clone SEGGER RTT and set `SEGGER_RTT_PATH` to the checkout root containing `RTT/SEGGER_RTT.c`. |
| AOT says `helia-aot is not installed` | Install the AOT extra: `uv sync --extra aot` or `pip install 'helia-profiler[aot]'`. |
| Firmware reports arena/memory failure | Increase the RT arena size with `with_model(..., arena_size=...)`, or review AOT placement settings. |
| PMU overflow warning | Reduce iteration count, change counters, or treat affected values as unreliable. |
| Compare says input is not a directory | Pass completed `ProfileResult` values or directories containing the standard reports. |

A good support bundle is the full result directory plus the resolved session configuration used to generate it.

## 11. Next Steps

Branch the immutable base session for one change at a time and keep result directories separate:

- `helia-rt` vs `helia-aot`
- GCC vs ATfE or Arm Compiler
- Different PMU counter groups: `cpu`, `memory`, `mve`
- Different memory placement settings
- Power enabled vs disabled
- Different model revisions

Then retain the returned results and use `Session.compare()` plus Model Explorer overlays to understand what changed and where.